# Inventory Gap vs. Sales Velocity

This notebook answers the question: Which products have the widest gap between inventory levels and sales velocity?

It connects to the live PostgreSQL database, compares total stock with units sold, and highlights the largest inventory gaps in a table and inline chart.



# Inventory Gap vs. Sales Velocity

This notebook answers the question: Which products have the widest gap between inventory levels and sales velocity?

It connects to the live PostgreSQL database, calculates the gap between total stock and units sold, and shows the top products in a table and inline bar chart.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
from psycopg2.extras import RealDictCursor

DB_CONFIG = {
    'host': 'db',
    'port': 5432,
    'dbname': 'zava',
    'user': 'postgres',
    'password': 'P@ssw0rd!',
}


def fetch_query(query):
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(query)
            rows = cur.fetchall()
    finally:
        conn.close()
    return pd.DataFrame(rows)

query = """
WITH product_sales AS (
    SELECT
        p.product_id,
        p.product_name,
        SUM(oi.quantity) AS units_sold,
        SUM(oi.total_amount) AS sales_revenue
    FROM retail.order_items oi
    JOIN retail.products p
      ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.product_name
),
product_inventory AS (
    SELECT
        i.product_id,
        SUM(i.stock_level) AS total_stock
    FROM retail.inventory i
    GROUP BY i.product_id
)
SELECT
    ps.product_name,
    ps.units_sold,
    COALESCE(pi.total_stock, 0) AS total_stock,
    COALESCE(pi.total_stock, 0) - ps.units_sold AS inventory_gap,
    CASE
        WHEN ps.units_sold = 0 THEN 0
        ELSE (COALESCE(pi.total_stock, 0) / ps.units_sold)::numeric(12,2)
    END AS stock_to_sales_ratio
FROM product_sales ps
LEFT JOIN product_inventory pi
  ON pi.product_id = ps.product_id
ORDER BY ABS(COALESCE(pi.total_stock, 0) - ps.units_sold) DESC, ps.units_sold DESC
LIMIT 20;
"""

inventory_gap_df = fetch_query(query)
inventory_gap_df


In [ ]:
%matplotlib inline
chart_df = inventory_gap_df.head(10).copy()
chart_df['label'] = chart_df['product_name']

plt.figure(figsize=(12, 6))
plt.bar(chart_df['label'], chart_df['inventory_gap'], color='crimson')
plt.title('Largest Inventory Gap vs. Sales Velocity')
plt.xlabel('Product')
plt.ylabel('Inventory Gap (Total Stock - Units Sold)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()
